#  🍄 SUSTAIN 🍄
### A Practical Example
Inspired by the famous edible-mushroom problem, we aim to solve it using technology from the 80s! I this notebook
we are trying to classify the mushrooms based on their features.

Big shootout for the creators of the dataset: https://archive.ics.uci.edu/dataset/73/mushroom !

### (1) Preprocessing
We will load the dataset and then encode a few chosen features of mushrooms as vectors.


In [37]:
import pandas as pd
import os

In [38]:
print(f"working dir: {os.getcwd()}")

working dir: C:\Users\kalus\PycharmProjects\SUSTAIN-replication


In [39]:
df = pd.read_csv("notebooks/datasets/mushrooms.csv", sep=",", encoding="utf-8")
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8124 entries, 0 to 8123
Data columns (total 23 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   class                     8124 non-null   str  
 1   cap-shape                 8124 non-null   str  
 2   cap-surface               8124 non-null   str  
 3   cap-color                 8124 non-null   str  
 4   bruises                   8124 non-null   str  
 5   odor                      8124 non-null   str  
 6   gill-attachment           8124 non-null   str  
 7   gill-spacing              8124 non-null   str  
 8   gill-size                 8124 non-null   str  
 9   gill-color                8124 non-null   str  
 10  stalk-shape               8124 non-null   str  
 11  stalk-root                8124 non-null   str  
 12  stalk-surface-above-ring  8124 non-null   str  
 13  stalk-surface-below-ring  8124 non-null   str  
 14  stalk-color-above-ring    8124 non-null   str  
 15

In [40]:
# the most important variables are class
df.sample(4)

,class,cap-shape,cap-surface,cap-color,bruises,odor,gill-attachment,gill-spacing,gill-size,gill-color,...,stalk-surface-below-ring,stalk-color-above-ring,stalk-color-below-ring,veil-type,veil-color,ring-number,ring-type,spore-print-color,population,habitat
1918,e,f,f,n,f,n,f,w,b,k,...,f,w,w,p,w,o,e,k,s,g
2664,e,x,f,g,t,n,f,c,b,p,...,s,p,w,p,w,o,p,k,v,d
6504,p,f,y,e,f,y,f,c,n,b,...,s,w,p,p,w,o,e,w,v,d
1028,e,x,y,y,t,l,f,c,b,k,...,s,w,w,p,w,o,p,k,n,m


What we have to do now is to transform the output into the list readable for SUSTAIN.

### (2) Encoding
Now, we need to encode the stimuli as vectors suitable for SUSTAIN.

In [41]:
from src.expretimetnal_tools import encode_stimuli

# 70/30 split
split_index = int(len(df) * 0.70)
data_train = df.iloc[:split_index]
data_test = df.iloc[split_index:]

train_encoded_stimuli, dim_sizes_train = encode_stimuli(data_train)
test_encoded_stimuli, dim_sizes_test = encode_stimuli(data_test)

dim_sizes: [2, 6, 4, 10, 2, 8, 1, 2, 2, 10, 2, 5, 3, 4, 6, 7, 1, 1, 2, 4, 6, 6, 7]
dim_sizes: [2, 5, 4, 10, 2, 5, 2, 2, 2, 11, 2, 3, 4, 4, 8, 8, 1, 4, 3, 5, 7, 5, 7]


We must combine the dim_sizes. Why? SUSTAIN internal weights are built expecting up to n values. Thus, different dimensions are not a problem when working with lower dimensions than expected. However, they are a huge problem when working with higher dimensions than expected (e.g. when the train set has fewer dimensions than the test set).

In [42]:
dim_sizes_combined = [max(a, b) for a, b in zip(dim_sizes_train, dim_sizes_test)]
sustain_run = [train_encoded_stimuli, test_encoded_stimuli]
print(len(test_encoded_stimuli))
print(f"dim_sizes_combined: {dim_sizes_combined}")

2438
dim_sizes_combined: [2, 6, 4, 10, 2, 8, 2, 2, 2, 11, 2, 5, 4, 4, 8, 8, 1, 4, 3, 5, 7, 6, 7]


### (3) Test run
Let's see how SUSTAIN actually performs. We are going to run it in the following fashion: first let's SUSTAIN in supervised mode first in order to train its prediction of a given class: poisonous/nonpoinsoneuss.

We will test the model's accuracy through a series of iterations.

In [55]:
from importlib import reload
import src.expretimetnal_tools
reload(src.expretimetnal_tools)
from src.expretimetnal_tools import run_sustain_on_data
from src.sustain import SUSTAIN

model = SUSTAIN(r=2.844642,
                beta=2.386305,
                d=12.0,
                eta=0.09361126,
                supervised=True,
                queried_dim=0)

model.reset(dim_sizes=dim_sizes_combined)

run_sustain_on_data(model=model, data=train_encoded_stimuli, queried_dim=0)
run_sustain_on_data(model=model, data=test_encoded_stimuli, queried_dim=0)

,response,probability,correct,n_clusters,winner,recruited
index,,,,,,
0,0,"[0.5, 0.5]",False,1,0,True
1,0,"[0.5, 0.5]",False,2,1,True
2,0,"[0.58055412583982, 0.41944587416018003]",True,2,1,False
3,1,"[0.35105598948061856, 0.6489440105193814]",True,2,0,False
4,0,"[0.6611945180581504, 0.33880548194184945]",True,2,1,False
...,...,...,...,...,...,...
5681,0,"[0.5000061287383812, 0.49999387126161876]",True,37,30,False
5682,1,"[0.4999813562027104, 0.5000186437972897]",True,37,23,False
5683,0,"[0.5000322845071082, 0.49996771549289176]",True,37,26,False


In [ ]:
results.head(5)

In [ ]:
count_true = (results["correct"] == True).sum()
count_false = (results["correct"] == False).sum()
accuracy = count_true / (count_false + count_true)
print(accuracy)

In [ ]:
count_false